In [1]:
import glob
import pandas as pd
excel_files = glob.glob("../../data/*.xls")
print(excel_files)

# Check if every columns are identical
headers = []

for f in excel_files:
    try:
        # Try reading as Excel
        df = pd.read_excel(f, engine="xlrd", nrows=1, header=None)
        header = df.iloc[0].tolist()
    except Exception as e:
        # Read as CSV
        df = pd.read_csv(f, engine="python", on_bad_lines='skip', sep="\t", nrows=1, header=None)
        header = df.iloc[0].tolist()
    headers.append((f, header))
    
# Combine all data into a single DataFrame    
df_list = []

for f in excel_files:
    try:
        #Try reading as Excel
        df = pd.read_excel(f, engine="xlrd")
        print(f"Read {f} as Excel")
    except Exception as e:
        #read as CSV
        df = pd.read_csv(f, engine="python", on_bad_lines='skip', sep="\t")
        print(f"Read {f} as CSV")
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)
print("DataFrames combined successfully")


['../../data\\2013-03.xls', '../../data\\2013-04.xls', '../../data\\2014-01.xls', '../../data\\2014-02.xls', '../../data\\2014-03.xls', '../../data\\2014-04.xls', '../../data\\2015-01.xls', '../../data\\2015-02.xls', '../../data\\2015-03.xls', '../../data\\2015-04.xls', '../../data\\2016-01.xls', '../../data\\2016-02.xls', '../../data\\2016-03.xls', '../../data\\2016-04.xls', '../../data\\2017-01.xls', '../../data\\2017-02.xls', '../../data\\2017-03.xls', '../../data\\2017-04.xls', '../../data\\2018-01.xls', '../../data\\2018-02.xls', '../../data\\2018-03.xls', '../../data\\2018-04.xls', '../../data\\2019-01.xls', '../../data\\2019-02.xls', '../../data\\2019-03.xls', '../../data\\2019-04.xls', '../../data\\2020-01.xls', '../../data\\2020-02.xls', '../../data\\2020-03.xls', '../../data\\2020-04.xls', '../../data\\2021-01.xls', '../../data\\2021-02.xls', '../../data\\2021-03.xls', '../../data\\2021-04.xls', '../../data\\2022-01.xls', '../../data\\2022-02.xls', '../../data\\2022-03.xls', 

In [3]:
columns = combined_df.columns.unique().tolist()
print(columns)

['Datum+Uhrzeit', 'URB Lysimeter outside tension kPa', 'Status', 'Datum+Uhrzeit.1', 'URB Lysimeter inside tension kPa', 'Status.1', 'Datum+Uhrzeit.2', 'URB Lysimeter outside tension reference kPa', 'Status.2', 'Datum+Uhrzeit.3', 'URB Lysimeter vacuum kPa', 'Status.3', 'Datum+Uhrzeit.4', 'URB Lysimeter level cm', 'Status.4', 'Datum+Uhrzeit.5', 'URB Lysimeter level reference cm', 'Status.5', 'Datum+Uhrzeit.6', 'URB Lysimeter percolation pump mV', 'Status.6', 'Datum+Uhrzeit.7', 'URB Lysimeter temperature control mV', 'Status.7', 'Datum+Uhrzeit.8', 'URB Lysimeter outside temperature degC', 'Status.8', 'Datum+Uhrzeit.9', 'URB Lysimeter inside temperature degC', 'Status.9', 'Datum+Uhrzeit.10', 'URB Lysimeter discharge (1) l', 'Status.10', 'Datum+Uhrzeit.11', 'SSA 3 Lysimeter tension 30cm kPa', 'Status.11', 'Datum+Uhrzeit.12', 'SSA 3 Lysimeter vacuum 30cm kPa', 'Status.12', 'Datum+Uhrzeit.13', 'SSA 3 Lysimeter tension 75cm kPa', 'Status.13', 'Datum+Uhrzeit.14', 'SSA 3 Lysimeter vacuum 75cm kP

In [14]:
import re
from collections import OrderedDict

# Get all current columns (assumes combined_df already created)
all_cols = combined_df.columns.astype(str).tolist()

# Function to remove trailing ".<digits>" suffix added by pandas
def base_name(col):
    return re.sub(r"\.\d+$", "", col)

seen = set()
cols_to_keep = []
# Keep first occurrence for each base name (preserve order)
for c in all_cols:
    b = base_name(c)
    if b not in seen:
        seen.add(b)
        cols_to_keep.append(c)

# Optionally drop generic columns (uncomment if needed)
# cols_to_keep = [c for c in cols_to_keep if base_name(c) != "Status"]

# Apply selection: keep first occurrence per base name
combined_df = combined_df.loc[:, cols_to_keep]

# Rename columns to their base names (remove suffixes)
new_names = {orig: base_name(orig) for orig in combined_df.columns}
combined_df = combined_df.rename(columns=new_names)

# Verify result
print(f"Columns after cleanup ({len(combined_df.columns)}):")
columns = combined_df.columns.unique().tolist()

# Option A: každý element na nový riadok, oddelené čiarkou (čiarka + newline)
print(*columns, sep=",\n")

Columns after cleanup (77):
Datum+Uhrzeit,
URB Lysimeter outside tension kPa,
Status,
URB Lysimeter inside tension kPa,
URB Lysimeter outside tension reference kPa,
URB Lysimeter vacuum kPa,
URB Lysimeter level cm,
URB Lysimeter level reference cm,
URB Lysimeter percolation pump mV,
URB Lysimeter temperature control mV,
URB Lysimeter outside temperature degC,
URB Lysimeter inside temperature degC,
URB Lysimeter discharge (1) l,
SSA 3 Lysimeter tension 30cm kPa,
SSA 3 Lysimeter vacuum 30cm kPa,
SSA 3 Lysimeter tension 75cm kPa,
SSA 3 Lysimeter vacuum 75cm kPa,
SSA 3 Lysimeter tension 120cm kPa,
SSA 3 Lysimeter vacuum 120cm kPa,
SSA 3 Lysimeter UMP 30cm %,
SSA 3 Lysimeter UMP 75cm %,
SSA 3 Lysimeter UMP 120cm %,
SSA 3 Lysimeter temperature 30cm degC,
SSA 3 Lysimeter temperature 75cm degC,
SSA 3 Lysimeter temperature 120cm degC,
SSA 3 Lysimeter ec 30cm mS/cm,
SSA 3 Lysimeter ec 75cm mS/cm,
SSA 3 Lysimeter ec 120cm mS/cm,
SSA 3 Lysimeter battery V,
SSA 3 Lysimeter scale (1) kg,
SSA 4 Schac